# **Handin 1.**

Implement a circular queue with a fixed size based on an array (you may use Pythons list which is a kind of ArrayList). The implementation must support the following operations:

| Operation | What it does | Return value (normal) | Return value (edge case) |
|---|---|---|---|
| `enqueue(x)` | Inserts x at the rear of the queue | True (success) | False (if the queue is full) |
| `dequeue()` | Removes and returns the front element | The element | None (if the queue is empty) |
| `front()` | Returns the front element without removing it | The element | None (if the queue is empty) |
| `is_empty()` | Checks whether the queue is empty | True / False | – |
| `is_full()` | Checks whether the queue is full | True / False | – |
| `size()` | Returns the number of elements in the queue | integer ≥ 0 | – |


All operations must be tested with comprehensive tests.

The handin must be a Jupyter Notebook file (.ipynb) that contains the application in one cell and the tests in another (all test must be executed by executing the cell with the tests).

In [3]:
class CircularQueue:
    def __init__(self, capacity):
        self.capacity = capacity
        self.data = [None] * capacity
        self.head = 0
        self.tail = 0
        self._size = 0

    def size(self):
        return self._size

    def is_empty(self):
        return self._size == 0

    def is_full(self):
        return self._size == self.capacity

    def enqueue(self, data):
        if self.is_full():
            return False
        else:
            self._size += 1
            self.data[self.tail] = data
            self.tail = (self.tail + 1) % self.capacity
            return True

    def dequeue(self):
        if self.is_empty():
            return None
        else:
            self._size -= 1
            dequeued_item = self.data[self.head]
            self.data[self.head] = None
            self.head = (self.head + 1) % self.capacity
            return dequeued_item

    def front(self):
        if self.is_empty():
            return None
        else:
            return self.data[self.head]




In [4]:
import unittest


class TestCircularQueue(unittest.TestCase):
    def setUp(self):
        self.q = CircularQueue(3)

    def fill(self, items):
        for item in items:
            self.q.enqueue(item)

    # --- new queue ---
    def test_new_queue_is_empty(self):
        self.assertTrue(self.q.is_empty())
        self.assertFalse(self.q.is_full())
        self.assertEqual(self.q.size(), 0)

    # --- enqueue ---
    def test_enqueue_returns_true(self):
        self.assertTrue(self.q.enqueue(1))

    def test_enqueue_increases_size(self):
        self.fill([1, 2])
        self.assertEqual(self.q.size(), 2)

    def test_enqueue_on_full_returns_false(self):
        self.fill([1, 2, 3])
        self.assertFalse(self.q.enqueue(4))

    def test_enqueue_on_full_does_not_change_queue(self):
        self.fill([1, 2, 3])
        self.q.enqueue(4)
        self.assertEqual(self.q.size(), 3)
        self.assertEqual([self.q.dequeue() for _ in range(3)], [1, 2, 3])

    # --- dequeue ---
    def test_dequeue_returns_front_element(self):
        self.fill([1, 2])
        self.assertEqual(self.q.dequeue(), 1)

    def test_dequeue_fifo_order(self):
        self.fill(["a", "b", "c"])
        self.assertEqual(self.q.dequeue(), "a")
        self.assertEqual(self.q.dequeue(), "b")
        self.assertEqual(self.q.dequeue(), "c")

    def test_dequeue_decreases_size(self):
        self.fill([1, 2])
        self.q.dequeue()
        self.assertEqual(self.q.size(), 1)

    def test_dequeue_on_empty_returns_none(self):
        self.assertIsNone(self.q.dequeue())

    def test_dequeue_after_emptying_returns_none(self):
        self.fill([1])
        self.q.dequeue()
        self.assertIsNone(self.q.dequeue())
        self.assertEqual(self.q.size(), 0)

    # --- front ---
    def test_front_returns_front_element(self):
        self.fill([1, 2])
        self.assertEqual(self.q.front(), 1)

    def test_front_does_not_remove(self):
        self.fill([1, 2])
        self.q.front()
        self.assertEqual(self.q.size(), 2)
        self.assertEqual(self.q.front(), 1)

    def test_front_after_dequeue(self):
        self.fill([1, 2])
        self.q.dequeue()
        self.assertEqual(self.q.front(), 2)

    def test_front_on_empty_returns_none(self):
        self.assertIsNone(self.q.front())

    # --- is_empty / is_full ---
    def test_is_empty_false_after_enqueue(self):
        self.q.enqueue(1)
        self.assertFalse(self.q.is_empty())

    def test_is_empty_true_after_emptying(self):
        self.fill([1, 2])
        self.q.dequeue()
        self.q.dequeue()
        self.assertTrue(self.q.is_empty())

    def test_is_full_true_at_capacity(self):
        self.fill([1, 2, 3])
        self.assertTrue(self.q.is_full())
        self.assertFalse(self.q.is_empty())

    def test_is_full_false_below_capacity(self):
        self.fill([1, 2])
        self.assertFalse(self.q.is_full())

    def test_is_full_false_after_dequeue_from_full(self):
        self.fill([1, 2, 3])
        self.q.dequeue()
        self.assertFalse(self.q.is_full())

    # --- size ---
    def test_size_tracks_every_operation(self):
        sizes = [self.q.size()]
        for item in [1, 2, 3, 4]:  # 4 is rejected (full)
            self.q.enqueue(item)
            sizes.append(self.q.size())
        for _ in range(4):  # last dequeue is on an empty queue
            self.q.dequeue()
            sizes.append(self.q.size())
        self.assertEqual(sizes, [0, 1, 2, 3, 3, 2, 1, 0, 0])

    # --- wrap-around ---
    def test_wrap_around_preserves_order(self):
        q = CircularQueue(5)
        for item in "ABCD":
            q.enqueue(item)
        q.dequeue()
        q.dequeue()
        for item in "EFG":  # tail wraps past the end of the array
            self.assertTrue(q.enqueue(item))
        self.assertTrue(q.is_full())
        self.assertFalse(q.enqueue("H"))
        self.assertEqual([q.dequeue() for _ in range(5)], list("CDEFG"))
        self.assertTrue(q.is_empty())

    def test_front_after_wrap_around(self):
        self.fill([1, 2, 3])
        self.q.dequeue()
        self.q.dequeue()
        self.q.dequeue()  # head wraps back to index 0
        self.q.enqueue(4)
        self.assertEqual(self.q.front(), 4)

    def test_many_cycles_match_reference_list(self):
        expected = []
        for i in range(100):
            if i % 3 == 2:
                self.assertEqual(self.q.dequeue(), expected.pop(0) if expected else None)
            else:
                self.assertEqual(self.q.enqueue(i), len(expected) < 3)
                if len(expected) < 3:
                    expected.append(i)
            self.assertEqual(self.q.size(), len(expected))
            self.assertEqual(self.q.front(), expected[0] if expected else None)

    # --- capacity 1 ---
    def test_capacity_one(self):
        q = CircularQueue(1)
        self.assertTrue(q.enqueue("x"))
        self.assertTrue(q.is_full())
        self.assertFalse(q.enqueue("y"))
        self.assertEqual(q.dequeue(), "x")
        self.assertTrue(q.is_empty())
        self.assertTrue(q.enqueue("z"))
        self.assertEqual(q.front(), "z")


unittest.main(argv=[""], verbosity=2, exit=False)


test_capacity_one (__main__.TestCircularQueue.test_capacity_one) ... ok
test_dequeue_after_emptying_returns_none (__main__.TestCircularQueue.test_dequeue_after_emptying_returns_none) ... ok
test_dequeue_decreases_size (__main__.TestCircularQueue.test_dequeue_decreases_size) ... ok
test_dequeue_fifo_order (__main__.TestCircularQueue.test_dequeue_fifo_order) ... ok
test_dequeue_on_empty_returns_none (__main__.TestCircularQueue.test_dequeue_on_empty_returns_none) ... ok
test_dequeue_returns_front_element (__main__.TestCircularQueue.test_dequeue_returns_front_element) ... ok
test_enqueue_increases_size (__main__.TestCircularQueue.test_enqueue_increases_size) ... ok
test_enqueue_on_full_does_not_change_queue (__main__.TestCircularQueue.test_enqueue_on_full_does_not_change_queue) ... ok
test_enqueue_on_full_returns_false (__main__.TestCircularQueue.test_enqueue_on_full_returns_false) ... ok
test_enqueue_returns_true (__main__.TestCircularQueue.test_enqueue_returns_true) ... ok
test_front_aft